# Caso F · 02 Tracking de experimentos con MLflow local

> _Tutorial · Caso de uso: **F — MLOps** · Capa Medallion: **transversal** · Spec: `docs/specs/synthetic-bms/01-product-spec.md`_

Material docente del proyecto **CAPTIA Synthetic Data BMS** — IES Dr. Lluís Simarro,
Curso de Especialización IA & Big Data 2025-2026.


## 1. Objetivo

Ejecutar un run completo del baseline del Caso B con MLflow local (SQLite) y demostrar la trazabilidad.


## 2. Qué se aprena

- `mlflow.start_run()`.
- `mlflow.log_param`, `mlflow.log_metric`, `mlflow.log_artifact`.
- `mlflow.set_tag("lakefs_tag", "...")`.


## 3. Contexto del caso de uso

Sin servidor MLflow externo: usamos backend `sqlite:///mlflow.db` y almacenamiento local.


## 4. Relación con CENTINELA+

Producción cambiará la URL del tracking server; el resto del código no.


## 5. Relación con Medallion

Transversal.


## 6. Datos de entrada

Mock BDG2.


## 7. Setup y variables de entorno

Cargamos las variables de entorno (`.env`), inicializamos `numpy` con `seed=42` y aplicamos el estilo de plotting compartido. Los helpers viven en `notebooks/_common/`.


In [1]:
# Setup canónico — todos los notebooks didácticos lo usan
from __future__ import annotations

import os
import sys
from pathlib import Path

ROOT = Path.cwd()
while ROOT.name and not (ROOT / "pyproject.toml").exists():
    ROOT = ROOT.parent
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from notebooks._common.captia_schema import (
    CANONICAL_TAGS, MEASUREMENT_TELEMETRY, MEASUREMENT_FAULT_LABELS,
    DEFAULT_BUCKET_RETENTIONS, KNOWN_VARIABLES,
    build_topic, build_line_protocol, validate_canonical_tags,
)
from notebooks._common.connection import load_env, get_influx_client, get_default_bucket
from notebooks._common.plotting import setup_default_style, plot_timeseries, plot_distribution
import notebooks._common.synthetic_mocks as mocks

SEED = 42
rng = np.random.default_rng(SEED)
setup_default_style()
load_env()
print(f"ROOT={ROOT}, SEED={SEED}, default_bucket={get_default_bucket()}")


ROOT=C:\CAPTIA\CAPTIA-SYNTHETIC-DATA-BMS, SEED=42, default_bucket=telemetry


## 8. Schema CAPTIA esperado

No aplica.


## 9. Carga de datos o mock

Cargamos features Caso B (regenerar si falta).


In [2]:
df, _ = mocks.make_bdg2_education_subset()
df = df[df.building_id == df.building_id.unique()[0]].set_index("timestamp")
X = pd.DataFrame({
    "y": df["power_kw"],
    "t_outdoor": df["t_outdoor"],
    "lag_24h": df["power_kw"].shift(24),
}).dropna()
y = X.pop("y")
n = len(X); i = int(n * 0.8)
X_tr, X_te = X.iloc[:i], X.iloc[i:]
y_tr, y_te = y.iloc[:i], y.iloc[i:]
print(len(X_tr), len(X_te))


6892 1724


## 10. Exploración paso a paso

Comprobamos si MLflow está disponible.


In [3]:
import os

try:
    import mlflow
    HAS_MLFLOW = True
except ImportError:
    HAS_MLFLOW = False
print("mlflow disponible:", HAS_MLFLOW)


mlflow disponible: True


## 11. Transformación bronce → plata

No aplica.


## 12. Construcción de capa oro

**Tres runs comparables** para que el alumno vea MLflow funcionando: (1) baseline naïve-24h sin entrenamiento, (2) RF con 100 árboles, (3) RF con 300 árboles + max_depth 8. Loggea params + métricas + el improvement vs naïve para cada uno.


In [4]:
import math
from sklearn.ensemble import RandomForestRegressor
from notebooks._common.eval_helpers import naive_persistence_24h, mae as _mae, rmse as _rmse

assert HAS_MLFLOW, "MLflow es obligatorio para este notebook (`uv sync --group notebooks`)"

mlflow_dir = ROOT / "output" / "mlruns"
mlflow_dir.mkdir(parents=True, exist_ok=True)
mlflow.set_tracking_uri(mlflow_dir.as_uri())
mlflow.set_experiment("case_B_baseline_2026")

# Baseline naïve-24h (referencia obligatoria)
y_naive = naive_persistence_24h(y_tr, y_te)
mae_naive = _mae(y_te.to_numpy(), y_naive)

runs = [
    {"name": "rf_n100_d5",  "n_estimators": 100, "max_depth": 5},
    {"name": "rf_n300_d8",  "n_estimators": 300, "max_depth": 8},
]

run_ids = []
for cfg in runs:
    with mlflow.start_run(run_name=cfg["name"]) as run:
        mlflow.log_params({**cfg, "seed": SEED})
        m = RandomForestRegressor(
            n_estimators=cfg["n_estimators"], max_depth=cfg["max_depth"],
            random_state=SEED, n_jobs=1,
        ).fit(X_tr, y_tr)
        y_pred = m.predict(X_te)
        mae_m = _mae(y_te.to_numpy(), y_pred)
        rmse_m = _rmse(y_te.to_numpy(), y_pred)
        improvement = (1 - mae_m / mae_naive) * 100 if mae_naive > 0 else 0
        mlflow.log_metric("MAE", mae_m)
        mlflow.log_metric("RMSE", rmse_m)
        mlflow.log_metric("MAE_naive", mae_naive)
        mlflow.log_metric("MAE_improvement_pct", improvement)
        mlflow.set_tag("lakefs_tag", "case_B/baseline_v1")
        mlflow.set_tag("baseline", "naive_persistence_24h")
        # artefact plot
        plt.figure(figsize=(10, 3))
        plt.plot(y_te.index, y_te.values, label="real", color="#3F51B5", linewidth=1)
        plt.plot(y_te.index, y_pred, label="pred", color="#FF5722", linewidth=1, alpha=0.9)
        plt.plot(y_te.index, y_naive, label="naive_24h", color="gray", linewidth=0.7, alpha=0.7)
        plt.legend(loc="upper right", fontsize=8)
        plt.title(f"{cfg['name']}: MAE={mae_m:.2f} (vs naive={mae_naive:.2f}, +{improvement:.1f}%)")
        plot_path = ROOT / "output" / f"case_F_{cfg['name']}.png"
        plt.savefig(plot_path, dpi=120, bbox_inches="tight")
        plt.close()
        mlflow.log_artifact(str(plot_path))
        run_ids.append(run.info.run_id)
        print(f"  {cfg['name']}: MAE={mae_m:.2f}  RMSE={rmse_m:.2f}  vs naive +{improvement:.1f}%")
print(f"\n{len(run_ids)} runs registrados en {mlflow_dir.as_uri()}")


C:\CAPTIA\CAPTIA-SYNTHETIC-DATA-BMS\.venv\Lib\site-packages\mlflow\tracking\_tracking_service\utils.py:184: FutureWarning: The filesystem tracking backend (e.g., './mlruns') is deprecated as of February 2026. Consider transitioning to a database backend (e.g., 'sqlite:///mlflow.db') to take advantage of the latest MLflow features. See https://mlflow.org/docs/latest/self-hosting/migrate-from-file-store for migration guidance.
  return FileStore(store_uri, store_uri)


  rf_n100_d5: MAE=4.45  RMSE=6.07  vs naive +10.8%


  rf_n300_d8: MAE=4.43  RMSE=6.08  vs naive +11.2%

2 runs registrados en file:///C:/CAPTIA/CAPTIA-SYNTHETIC-DATA-BMS/output/mlruns


## 13. Visualizaciones explicativas

Listamos los runs en la experiment con `mlflow.search_runs()` y comparamos métricas en una tabla.


In [5]:
runs_df = mlflow.search_runs(experiment_names=["case_B_baseline_2026"], output_format="pandas")
cols = ["tags.mlflow.runName", "metrics.MAE", "metrics.RMSE",
        "metrics.MAE_improvement_pct", "params.n_estimators", "params.max_depth"]
present = [c for c in cols if c in runs_df.columns]
runs_summary = runs_df[present].sort_values("metrics.MAE")
print(runs_summary.to_string(index=False))


tags.mlflow.runName  metrics.MAE  metrics.RMSE  metrics.MAE_improvement_pct params.n_estimators params.max_depth
         rf_n300_d8     4.425750      6.078417                    11.210493                 300                8
         rf_n300_d8     4.425750      6.078417                    11.210493                 300                8
         rf_n300_d8     4.425750      6.078417                    11.210493                 300                8
         rf_n300_d8     4.425750      6.078417                    11.210493                 300                8
         rf_n300_d8     4.425750      6.078417                    11.210493                 300                8
         rf_n300_d8     4.425750      6.078417                    11.210493                 300                8
         rf_n300_d8     4.425750      6.078417                    11.210493                 300                8
         rf_n300_d8     4.425750      6.078417                    11.210493                 300 

## 14. Validaciones

Cada run debe (a) batir la línea naïve-24h en MAE y (b) tener `MAE_improvement_pct > 0`.


In [6]:
for run_id in run_ids:
    run = mlflow.get_run(run_id)
    mae_m = run.data.metrics["MAE"]
    impr = run.data.metrics["MAE_improvement_pct"]
    assert mae_m < mae_naive, f"Run {run.info.run_name} no bate naive ({mae_m:.2f} >= {mae_naive:.2f})"
    assert impr > 0, f"Improvement no positivo en {run.info.run_name}"
print("Validaciones OK — todos los runs baten naive-24h")


Validaciones OK — todos los runs baten naive-24h


## 15. Errores comunes

1. Olvidar `set_tracking_uri` — los runs se pierden en /tmp.
2. Subir el modelo en cada run sin necesidad — usar `register_model`.
3. No loggear el `seed`.


## 16. Ejercicios propuestos

1. Añade `mlflow.log_text(json.dumps(env))` para capturar versiones.
2. Compara dos runs en la UI (`mlflow ui`).
3. Convierte el run en un script reproducible.


## 17. Cómo se reutiliza con datos reales

Cambiar `tracking_uri` al servidor de producción.


## 18. Resumen final y próximos pasos

Recuerda los conceptos principales del notebook y enlaza al siguiente paso.

- Siguiente notebook: `06_case_F_mlops/03_reproducibilidad_datasets_modelos.ipynb`.
- Documento web del caso: `docs/use-cases/case-f-mlops.md`.


## 19. Marco teórico (nivel doctoral)

### Versionado de datasets (lakeFS)

Modelo Git-like sobre object storage con commits inmutables:

$$
\text{commit}_t = \langle \text{tree}_t, \text{parent}_{t-1}, \text{metadata}_t \rangle
$$

con `tree` Merkle DAG. Modelos referencian
$\text{dataset\_uri} = \text{lakefs://repo/branch/commit\_id}$ no paths mutables.

### MLflow Run Schema

$$
\text{run}_i = \langle \text{params}_i, \text{metrics}_i, \text{artifacts}_i, \text{tags}_i, \text{dataset\_uri}_i \rangle
$$

### Reproducibilidad determinista

$$
\hat{y} = M(D, \theta, s = 42), \quad
\text{hash}(\hat{y}_1) = \text{hash}(\hat{y}_2) \iff (D_1, \theta_1, s_1) = (D_2, \theta_2, s_2)
$$

(ADR-008). Verificable con `pytest -m snapshot`.

### Linaje de features

$$
\text{Feature}_i = f_i(\text{Feature}_j, \text{Feature}_k, ...) \implies \text{DAG}_{features}
$$

Trazabilidad bidireccional dataset $\leftrightarrow$ run $\leftrightarrow$ deploy.


## 20. Visión corporativa CAPTIA

### Propuesta de valor

MLOps no genera ROI directo, pero **reduce el coste de toda la cadena**. Permite a CAPTIA mover modelos a producción con confianza, hacer auditorías regulatorias (próxima EU AI Act) y evitar el clásico *funcionaba en mi máquina*.

### ROI estimado

| Concepto | Valor |
|---|---|
| Reducción tiempo auditoría modelos (8 h → 1 h) | +800 €/año |
| Eliminación re-runs por incertidumbre | +400 €/año compute |
| Cumplimiento EU AI Act (riesgo evitado) | +20 000 € (riesgo) |
| **Bruto** | **+1 200 €/año** + risk hedging |

> **Trazabilidad ROI:** las cifras de esta tabla son derivables de [`docs/captia/economic_baseline.md`](../../docs/captia/economic_baseline.md) Sec 4 (compliance EU AI Act). Si una cifra no aparece allí, NO se reporta aquí (política anti NA-E).


## 21. Bibliografía y referencias

- Zaharia, M. et al. (2018). *Accelerating the Machine Learning Lifecycle with MLflow*. CIDR.
- lakeFS Project. *Documentation*. https://docs.lakefs.io
- Sculley, D. et al. (2015). *Hidden Technical Debt in Machine Learning Systems*. NeurIPS.
- EU (2024). *Artificial Intelligence Act*. Regulation (EU) 2024/1689.


## 22. Etapa del pipeline · Tracking experimentos + lakeFS tag

Cada run debe llevar `mlflow.set_tag('lakefs_tag', 'case_B/baseline_v1')`. El día que llegue una auditoría EU AI Act, el responsable necesita **reproducir** el modelo desplegado hace 6 meses sobre los datos exactos — sin tag lakeFS, irreproducible.

> El ROI cuantificado de esta etapa está anclado en [`docs/captia/economic_baseline.md`](../../docs/captia/economic_baseline.md) — cualquier cifra de la sección 20 es derivable de ahí, no inventada.